In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import time

In [ ]:
# Carrega o arquivo .env
load_dotenv(override=True)

In [ ]:
with open('.env', 'r') as f:
    conteudo = f.read()
    print(conteudo)

In [ ]:
# 1. Define o endereço (URL) do endpoint da API do Bubble de onde vamos extrair os dados de transações
url = "https://myfin-financial-management.bubbleapps.io/api/1.1/obj/transactions/"

# 2. Vai buscar o token de acesso seguro guardado nas variáveis de ambiente e limpa eventuais espaços ou quebras de linha
token = os.getenv("TOKEN_API").strip()

# 3. Cria o cabeçalho HTTP necessário para a autenticação, injetando o token no formato padrão "Bearer" exigido pela API
headers = {"Authorization": f"Bearer {token}"}

In [ ]:

def chamar_api_myfinance(url):
    # Cria uma lista vazia para armazenar todos os registos vindos de todas as páginas
    lista_dados_todas_paginas = []
    # Inicializa o cursor a 0 para começar a busca a partir do primeiro registo
    cursor = 0
    
    # Inicia um loop infinito que só será interrompido quando acabarem os dados
    while True:
        # Faz o pedido GET à API passando a URL, os headers de autenticação e o cursor atual nos parâmetros
        response = requests.get(url, headers=headers, params={'cursor': cursor})

        # Converte a resposta bruta da API num dicionário Python (formato JSON)
        response_ajustado_json = response.json()
        # Extrai a chave "response" do JSON (retorna None se a chave não existir)
        dados_response = response_ajustado_json.get("response", None)
        
        # Verifica se o dicionário/dados da resposta existem de facto
        if dados_response is not None:
            # Extrai a lista de resultados da página atual (se não existir, retorna uma lista vazia)
            results = dados_response.get('results', [])
            
            # Extrai quantos registos ainda faltam ser buscados na API
            remaining = dados_response.get('remaining', 0)

            # Adiciona os itens da página atual à nossa lista principal acumulada
            lista_dados_todas_paginas.extend(results)

            # Se não restarem mais registos para buscar, interrompe o loop
            if remaining <= 0:
                break
        else:
            # Se a resposta vier vazia ou sem a chave "response", interrompe o loop por segurança
            break

        # Avança o cursor em 100 unidades (assumindo blocos de 100 em 100 registos por página)
        cursor += 100
        # Pausa a execução por 1 segundo entre as requisições para evitar bloqueios de taxa (rate limit) na API
        time.sleep(1)

    # Retorna a lista completa com todos os dados recolhidos de todas as páginas
    return lista_dados_todas_paginas

In [ ]:
lista_dados_todas_paginas = chamar_api_myfinance(url)

In [ ]:
print(lista_dados_todas_paginas)

In [ ]:
# Criando o dataframe
df = pd.DataFrame(lista_dados_todas_paginas)


df

In [ ]:
# Exportando para Excel e parquet
df.to_excel('Transactions.xlsx', index=None)
df.to_parquet('Transactions.parquet', index=None)